## Image

In [ ]:
from dotenv import load_dotenv
import os
import base64
from groq import Groq
from flask import Flask, request, jsonify

load_dotenv()

app = Flask(__name__)
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def encode_image_to_dataurl(file_bytes, mime_type="image/jpeg"):
    b64 = base64.b64encode(file_bytes).decode("utf-8")
    return f"data:{mime_type};base64,{b64}"

In [ ]:
@app.route("/caption", methods=["POST"])
def caption():
    """
    Endpoint expects multipart/form-data with field 'image'
    Returns JSON: { "caption": "..." }
    """
    img = request.files.get("image")
    if not img:
        return jsonify({"error": "No image uploaded"}), 400

    data_url = encode_image_to_dataurl(img.read(), img.mimetype)

    completion = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": "Generate a caption for this outfit image."},
                {"type": "image_url", "image_url": {"url": data_url}}
            ]
        }],
        temperature=0.7,
        max_completion_tokens=200,
        top_p=0.9,
        response_format={"type": "json_object"}
    )

    # ekstrak caption
    msg = completion.choices[0].message
    # jika JSON mode, field mungkin di .content["caption"]
    try:
        caption_text = msg.content.get("caption") or msg.content.get("text") or msg
    except:
        caption_text = str(msg)

    return jsonify({"caption": caption_text})

if __name__ == "__main__":
    app.run(debug=True)

LangChain adalah framework (kerangka kerja) sumber terbuka (open-source) yang dirancang untuk mengembangkan aplikasi yang menggunakan model bahasa besar (Large Language Model/LLM). Framework ini menyediakan komponen-komponen dan integrasi yang dapat digunakan untuk membangun aplikasi-aplikasi yang canggih dengan memanfaatkan kemampuan model bahasa besar.

LangChain dikembangkan untuk membantu para pengembang dalam menciptakan aplikasi yang dapat berinteraksi dengan pengguna secara lebih alami dan efektif dengan menggunakan kekuatan model bahasa. Dengan menggunakan LangChain, pengembang dapat lebih fokus pada logika aplikasi dan pengalaman pengguna, sementara framework tersebut menangani integrasi dengan model bahasa besar.

Beberapa fitur utama LangChain meliputi:

1. **Komponen Modular**: LangChain menyediakan berbagai komponen yang dapat digunakan secara modular untuk membangun aplikasi. Ini termasuk komponen untuk pengolahan bahasa alami, pengambilan data, dan interaksi dengan model

## Web Search

In [21]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
import os
from langchain.tools import Tool
from dotenv import load_dotenv
load_dotenv()

True

In [22]:
llm = ChatGroq(model="meta-llama/llama-4-maverick-17b-128e-instruct", temperature=0)
tavily_search = TavilySearchResults(api_key=os.getenv("TAVILY_API_KEY"))

In [23]:
prompt_template = """You are a fashion stylist AI assistant. Your task is to recommend a complete outfit based on the user's request and provide purchase links for each item using the available search tool.
You MUST respond ONLY with a valid JSON object containing the outfit recommendation. Do not include any introductory text, explanations, or closing remarks outside the JSON structure.

User's request: {user_input}

Generate a JSON object following this exact structure:
{
    "products": [
        {
            "category": "top",
            "name": "<suggested top name>",
            "links": [
                "<link1>",
                "<link2>"
            ]
        },
        {
            "category": "bottom",
            "name": "<suggested bottom name>",
            "links": [
                "<link1>",
                "<link2>"
            ]
        },
        {
            "category": "shoes",
            "name": "<suggested shoes name>",
            "links": [
                "<link1>",
                "<link2>"
            ]
        },
        {
            "category": "headwear",
            "name": "<suggested headwear name or 'None' if not applicable>",
            "links": [
                "<link1>",
                "<link2>"
            ]
        },
        {
            "category": "accessories",
            "name": "<suggested accessory name or 'None' if not applicable>",
            "links": [
                "<link1>",
                "<link2>"
            ]
        },
        {
            "category": "bag",
            "name": "<suggested bag name or 'None' if not applicable>",
            "links": [
                "<link1>",
                "<link2>"
            ]
        }
    ]
}

Instructions for JSON content:
- Include items for the categories: 'top', 'bottom', 'shoes', 'headwear', 'accessories', 'bag'.
- If a category like 'headwear', 'accessories', or 'bag' is not suitable for the outfit or request, set the 'name' to "None" and provide empty strings or placeholder links in the 'links' list.
- For each recommended item ('name'), use the search tool to find two relevant online shopping links and place them in the 'links' list.
- Ensure the final output is a single, valid JSON object and nothing else.
"""

In [24]:
search_specs_tool = Tool(
    name="SearchProductSpecs",
    func=TavilySearchResults(max_results=5, search_depth= 'basic'),
    description=prompt_template,
    max_retries=5,
)

agent_executor = create_react_agent(llm, [search_specs_tool])

In [25]:
user_input = " What should I wear for a night out?"
response = agent_executor.invoke({"messages": user_input})
print(response['messages'][-1].content)

{
"products": [
{
"category": "top",
"name": "Sequined Top",
"links": [
"https://www.prettylittlething.com/sequined-top-in-black/p6010024?_loc=UK&_m=product&_s=product&_a=search",
"https://www.windsorstore.com/products/sequin-cropped-top-in-black-502206"
]
},
{
"category": "bottom",
"name": "High-Waisted Jeans",
"links": [
"https://www.prettylittlething.com/high-waisted-jeans/p6020024?_loc=UK&_m=product&_s=product&_a=search",
"https://www.windsorstore.com/products/high-waisted-jeans-in-blue-502206"
]
},
{
"category": "shoes",
"name": "Black Heels",
"links": [
"https://www.prettylittlething.com/black-heels/p6030024?_loc=UK&_m=product&_s=product&_a=search",
"https://www.windsorstore.com/products/black-heels-502206"
]
},
{
"category": "headwear",
"name": "None",
"links": [
"",
""
]
},
{
"category": "accessories",
"name": "Statement Earrings",
"links": [
"https://www.prettylittlething.com/statement-earrings/p6040024?_loc=UK&_m=product&_s=product&_a=search",
"https://www.windsorstore.com/pr